In [0]:
# ─── CÉLULA 1 — CONFIGURAÇÃO, IMPORTS E EXPERIMENT MLFLOW ────────────────────────
# Configura o ambiente de rastreio de experimentos MLflow com Unity Catalog.
# mlflow.set_registry_uri("databricks-uc") é OBRIGATÓRIO antes de qualquer operação com o Model Registry no Databricks Serverless — sem isso, o MLflow tentará usar o registry legado (workspace-based) incompatível com Unity Catalog.
# MLFLOW_DFS_TMP redireciona artefatos temporários para o Volume, pois o DBFS root está desabilitado neste ambiente Serverless.

import mlflow
import mlflow.spark
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
import os

mlflow.set_registry_uri("databricks-uc")
os.environ['MLFLOW_DFS_TMP'] = "/Volumes/workspace/default/modelos_ml"

CATALOG = "workspace"
SCHEMA  = "default"

df_gold = spark.read.table(f"{CATALOG}.{SCHEMA}.telco_gold")

mlflow.set_experiment("/Users/seu-email/telco-churn-experiment")

print(f"✔ Ouro carregada: {df_gold.count()} linhas")
print(f"✔ MLflow experiment configurado")
print(f"✔ Registry URI: databricks-uc (Unity Catalog)")

In [0]:
# ─── CÉLULA 2 — SPLIT TREINO/TESTE ───────────────────────────────────────────────
# Divide o dataset em 80% treino e 20% teste com seed=42 para reprodutibilidade.
# O mesmo seed é usado no notebook 06 para garantir que o conjunto de teste na avaliação seja idêntico ao usado aqui — condição essencial para comparação justa de métricas entre os dois notebooks.
# A proporção de churn em cada split é verificada para confirmar que o desbalanceamento foi preservado (com 7.043 linhas a distribuição é representativa).

df_train, df_test = df_gold.randomSplit([0.8, 0.2], seed=42)

print(f"✔ Treino : {df_train.count()} linhas")
print(f"✔ Teste  : {df_test.count()} linhas")

pct_train = df_train.filter("label = 1").count() / df_train.count()
pct_test  = df_test.filter("label = 1").count() / df_test.count()
print(f"✔ % churn treino : {pct_train:.1%}")
print(f"✔ % churn teste  : {pct_test:.1%}")

In [0]:
# ─── CÉLULA 3 — AVALIADORES E FUNÇÃO DE TREINAMENTO ──────────────────────────────
# Define três avaliadores e uma função reutilizável que encapsula o ciclo completo:
# treino → predição → métricas → log MLflow.

# AUC-ROC (BinaryClassificationEvaluator) é a métrica principal por ser robusta ao desbalanceamento de classes (~26% churn). F1 e Accuracy são métricas secundárias.
# mlflow.start_run() cria um contexto de rastreio: todos os parâmetros, métricas e artefatos logados dentro do bloco ficam associados a esse run específico, permitindo comparação visual no MLflow UI.

evaluator_auc = BinaryClassificationEvaluator(metricName="areaUnderROC")
evaluator_f1  = MulticlassClassificationEvaluator(metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(metricName="accuracy")

def treinar_e_logar(modelo, params: dict, nome_run: str):
    with mlflow.start_run(run_name=nome_run):
        modelo_treinado = modelo.fit(df_train)
        preds = modelo_treinado.transform(df_test)

        auc = evaluator_auc.evaluate(preds)
        f1  = evaluator_f1.evaluate(preds)
        acc = evaluator_acc.evaluate(preds)

        mlflow.log_params(params)
        mlflow.log_metric("auc_roc", auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("accuracy", acc)
        mlflow.spark.log_model(modelo_treinado, "model")

        print(f"  [{nome_run}] AUC={auc:.4f}  F1={f1:.4f}  ACC={acc:.4f}")
        return modelo_treinado, preds

print("✔ Função treinar_e_logar definida")

In [0]:
# ─── CÉLULA 4 — MODELO 1: REGRESSÃO LOGÍSTICA (BASELINE) ─────────────────────────
# Regressão Logística serve como baseline linear e é altamente interpretável.
# regParam=0.01 aplica regularização L2 leve para evitar overfitting sem prejudicar a capacidade preditiva em datasets de dimensão moderada.
# elasticNetParam=0.0 = 100% L2 (Ridge) — sem sparsidade forçada nas features.
# maxIter=100 é suficiente para convergência neste dataset.

lr_params = {
    "model"          : "LogisticRegression",
    "maxIter"        : 100,
    "regParam"       : 0.01,
    "elasticNetParam": 0.0
}

lr = LogisticRegression(
    maxIter=lr_params["maxIter"],
    regParam=lr_params["regParam"],
    elasticNetParam=lr_params["elasticNetParam"],
    labelCol="label",
    featuresCol="features"
)

lr_model, lr_preds = treinar_e_logar(lr, lr_params, "logistic_regression")

In [0]:
# ─── CÉLULA 5 — MODELO 2: RANDOM FOREST ──────────────────────────────────────────
# Ensemble de 100 árvores de decisão treinadas em paralelo (bagging).
# Mais robusto a outliers e capaz de capturar interações não-lineares que a Regressão Logística não detecta.
# maxDepth=8 controla a complexidade de cada árvore; seed=42 garante reprodutibilidade na seleção aleatória de features e amostras bootstrap.

rf_params = {
    "model"   : "RandomForest",
    "numTrees": 100,
    "maxDepth": 8,
    "seed"    : 42
}

rf = RandomForestClassifier(
    numTrees=rf_params["numTrees"],
    maxDepth=rf_params["maxDepth"],
    seed=rf_params["seed"],
    labelCol="label",
    featuresCol="features"
)

rf_model, rf_preds = treinar_e_logar(rf, rf_params, "random_forest")

In [0]:
# ─── CÉLULA 6 — MODELO 3: GRADIENT BOOSTED TREES ─────────────────────────────────
# GBT constrói árvores sequencialmente, cada uma corrigindo os erros da anterior.
# stepSize=0.1 é o learning rate — valores menores reduzem overfitting mas exigem mais iterações para convergir.
# maxIter=50 é suficiente para datasets de médio porte como este.
# Tipicamente o mais preciso dos três, porém menos interpretável e mais lento.

gbt_params = {
    "model"   : "GBT",
    "maxIter" : 50,
    "maxDepth": 5,
    "stepSize": 0.1,
    "seed"    : 42
}

gbt = GBTClassifier(
    maxIter=gbt_params["maxIter"],
    maxDepth=gbt_params["maxDepth"],
    stepSize=gbt_params["stepSize"],
    seed=gbt_params["seed"],
    labelCol="label",
    featuresCol="features"
)

gbt_model, gbt_preds = treinar_e_logar(gbt, gbt_params, "gbt")

In [0]:
# ─── CÉLULA 7 — SELEÇÃO DO MODELO CAMPEÃO ────────────────────────────────────────
# Compara os três modelos pelo AUC-ROC no conjunto de teste e elege o campeão.
# AUC-ROC é a métrica de seleção por ser insensível ao threshold de decisão e robusta ao desbalanceamento de classes (~26% churn).
# O modelo campeão (LogisticRegression nesta execução) é armazenado no dict `campeao` para uso direto na célula de registro do Model Registry.

resultados = [
    {"modelo": "LogisticRegression", "auc": evaluator_auc.evaluate(lr_preds),
     "f1": evaluator_f1.evaluate(lr_preds),  "obj": lr_model},
    {"modelo": "RandomForest",       "auc": evaluator_auc.evaluate(rf_preds),
     "f1": evaluator_f1.evaluate(rf_preds),  "obj": rf_model},
    {"modelo": "GBT",               "auc": evaluator_auc.evaluate(gbt_preds),
     "f1": evaluator_f1.evaluate(gbt_preds), "obj": gbt_model},
]

campeao = max(resultados, key=lambda x: x["auc"])

print("\n── Ranking por AUC-ROC ──────────────────")
for r in sorted(resultados, key=lambda x: x["auc"], reverse=True):
    flag = " 🏆" if r["modelo"] == campeao["modelo"] else ""
    print(f"  {r['modelo']:22} AUC={r['auc']:.4f}  F1={r['f1']:.4f}{flag}")
print(f"\n✔ Modelo campeão: {campeao['modelo']}")

In [0]:
# ─── CÉLULA 8 — REGISTRO DO MODELO CAMPEÃO NO UNITY CATALOG ──────────────────────
# Registra o modelo campeão no MLflow Model Registry com Unity Catalog.
# A signature (infer_signature) define o contrato de entrada/saída do modelo: input = array de features, output = prediction + probability. Isso habilita validação automática no serving e documentação no catálogo.
# vector_to_array() é necessário pois o MLflow não aceita o tipo VectorUDT do Spark ML — converte para array nativo antes de enviar ao pandas.
# O modelo ficará acessível em: Catalog → workspace → default → Models.

from mlflow.models.signature import infer_signature
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

MODEL_NAME = f"{CATALOG}.{SCHEMA}.telco-churn-predictor"

sample_preds = campeao["obj"].transform(df_train)

sample_input = (
    sample_preds
    .withColumn("features", vector_to_array(F.col("features")))
    .select("features")
    .limit(100)
    .toPandas()
)

sample_output = (
    sample_preds
    .withColumn("prediction", F.col("prediction"))
    .withColumn("probability", vector_to_array(F.col("probability")))
    .select("prediction", "probability")
    .limit(100)
    .toPandas()
)

signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_name="register_champion"):
    mlflow.log_param("champion_model", campeao["modelo"])
    mlflow.log_metric("champion_auc",  campeao["auc"])
    mlflow.log_metric("champion_f1",   campeao["f1"])

    mlflow.spark.log_model(
        campeao["obj"],
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        signature=signature
    )

print(f"✔ Modelo registrado como '{MODEL_NAME}'")
print(f"✔ Signature: {signature}")
print(f"✔ Acesse: Catalog → workspace → default → Models → telco-churn-predictor")